In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

In [2]:
#Step 1: Load Dataset and Preprocessing
batch_size=64
transform=transforms.Compose([transforms.Resize((32,32)),
                              transforms.ToTensor(),#pixel is in [0,1]
                              transforms.Normalize((0.5,),(0.5,))])

train_dataset=torchvision.datasets.MNIST(root='./data',train=True,transform=transform,download=True)
test_dataset=torchvision.datasets.MNIST(root='./data',train=False,transform=transform,download=True)
train_loader=DataLoader(dataset=train_dataset,batch_size=batch_size,shuffle=True)
test_loader=DataLoader(dataset=test_dataset,batch_size=batch_size,shuffle=False)

100%|██████████| 9.91M/9.91M [00:00<00:00, 19.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 496kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.18MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.1MB/s]


In [3]:
#Step2: Define Model
#LeNet Architecture
class LeNet5(nn.Module):
   def __init__(self):
      super(LeNet5,self).__init__()
      self.conv_layer=nn.Sequential(
            nn.Conv2d(in_channels=1,out_channels=6,kernel_size=5,stride=1),
            nn.ReLU(),
            nn.AvgPool2d(kernel_size=2,stride=2),
            nn.Conv2d(in_channels=6,out_channels=16,kernel_size=5,stride=1),
            nn.ReLU(),
            nn.AvgPool2d(kernel_size=2,stride=2))
      self.fc_layer=nn.Sequential(
            nn.Linear(in_features=400,out_features=120),
            nn.ReLU(),
            nn.Linear(in_features=120,out_features=84),
            nn.ReLU(),
            nn.Linear(in_features=84,out_features=10))

   def forward(self,x):
      x=self.conv_layer(x)
      x=x.view(-1,16*5*5)#Flatten the tensor
      x=self.fc_layer(x)
      return(x)


In [4]:
#Step 3: Hyperparameter Setup
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
learning_rate=0.001
num_epochs=5

In [5]:
#Step 4: Loss function and Optimizer
model=LeNet5().to(device)
criterion=nn.CrossEntropyLoss()
optimizer=optim.Adam(model.parameters(),lr=learning_rate)

In [6]:
#Step 5: Training loop
print(f"Training on {device}")
model.train()
for epoch in range(num_epochs):
   loss=0
   for i,(images,labels) in enumerate(train_loader):
      images,labels=images.to(device), labels.to(device)
      #Forward pass
      output=model(images)
      loss=criterion(output,labels)
      #Backpropagation
      optimizer.zero_grad()
      loss.backward()#Evaluate gradient wrt loss
      optimizer.step()#Update weights
      loss=loss+loss.item()
   print(f"Epoch:{epoch+1},loss:{loss/len(train_loader)}")

Training on cuda
Epoch:1,loss:0.0003020139702130109
Epoch:2,loss:0.00015427620382979512
Epoch:3,loss:4.489280854613753e-06
Epoch:4,loss:2.030864379776176e-05
Epoch:5,loss:3.1365383620141074e-05


In [8]:
#Step 6: Evaluation
model.eval()
correct=0
total=0

with torch.no_grad():
   for images,labels in test_loader:
      images, labels = images.to(device), labels.to(device) # Move data to the same device as the model
      output=model(images)
      _,predicted=torch.max(output.data,1)
      total=total+labels.size(0)
      correct=correct+(predicted==labels).sum().item()
print(f"Test Accuracy:{correct/total}")

Test Accuracy:0.9896
